# Final Stage F: 18-Mask Coordinate and Regional-Ownership Comparator

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook reuses the `mask.npy` files already saved by the 51R2 and 53R2 call-capture runs. It does not regenerate masks or rerun Telea. It isolates the 9 exceptional masks on each side, pairs them within the seven failing pages, compares exact coordinates, and maps each differing coordinate to persisted Notebook45 regional `M3.png` ownership.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scipy


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, json, platform, sys, re
import cv2, numpy as np, pandas as pd
from scipy.optimize import linear_sum_assignment

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_51_OVERRIDE=None
CAPTURE_53_OVERRIDE=None
NOTEBOOK45_ROOT_OVERRIDE=None
PAGES={'Page003','Page004','Page005','Page008','Page036','Page040','Page049'}
EXPECTED=9
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_stage_f_18_mask_coordinate_comparison'
OUT=ROOT/'stage_f_18_mask_coordinate_comparison'/RUN
TABLES=OUT/'tables'; IMAGES=OUT/'images'; AUDIT=OUT/'audit'
for d in [OUT,TABLES,IMAGES,AUDIT]: d.mkdir(parents=True,exist_ok=False)

def req(x,msg):
    if not x: raise RuntimeError(msg)
def ah(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def bh(a): return ah((np.asarray(a)>0).astype(np.uint8))
def pageof(v):
    m=re.search(r'Page\d+',str(v),re.I)
    return None if not m else 'Page'+m.group(0)[4:].zfill(3)
def metafiles(root): return sorted(p for p in Path(root).rglob('*.json') if p.name in {'metadata.json','call_metadata.json'})
def sideof(root):
    p=Path(root)/'capture_complete.json'; text=''
    if p.exists():
        try: text=json.dumps(json.loads(p.read_text())).upper()
        except: pass
    text += ' '+str(root).upper()
    if '51R2' in text:return '51R2'
    if '53R2' in text:return '53R2'
    return None
def resolve_capture(side,override):
    if override:
        p=Path(override); req(p.exists() and metafiles(p),f'Invalid {side} override: {p}'); return p
    hits=[]
    for c in ROOT.rglob('capture_complete.json'):
        r=c.parent
        if sideof(r)==side and metafiles(r): hits.append((r.stat().st_mtime,len(metafiles(r)),r))
    req(hits,f'No completed {side} capture found. Set its override.')
    hits.sort(); print(side,'candidates:'); [print(' ',x[2],x[1]) for x in hits[-10:]]
    return hits[-1][2]

def pick(m,names):
    for k in names:
        if k in m and m[k] not in [None,'']: return m[k]
    c=m.get('context',{}) if isinstance(m.get('context'),dict) else {}
    for k in names:
        if k in c and c[k] not in [None,'']: return c[k]
    return None

def index(root,side):
    rows=[]; bad=[]
    for mp in metafiles(root):
        try:
            m=json.loads(mp.read_text()); page=pageof(pick(m,['page','page_id','page_name']))
            if page not in PAGES: continue
            flags=pick(m,['flags']); fname=str(pick(m,['flags_name']) or '')
            if flags is not None and int(flags)!=int(cv2.INPAINT_TELEA): continue
            if flags is None and 'TELEA' not in fname.upper(): continue
            mask=mp.parent/'mask.npy'
            if not mask.exists(): bad.append({'side':side,'metadata':str(mp),'error':'mask.npy missing'}); continue
            rows.append({'side':side,'page':page,'region_id':pick(m,['region_id','region','rid']),
                'call_number':int(pick(m,['call_number']) or -1),'call_dir':mp.parent,'mask_path':mask,
                'mask_hash':pick(m,['mask_boolean_sha256']),'mask_pixels':pick(m,['mask_pixels'])})
        except Exception as e: bad.append({'side':side,'metadata':str(mp),'error':repr(e)})
    if bad: pd.DataFrame(bad).to_csv(AUDIT/f'{side}_metadata_failures.csv',index=False)
    req(rows,f'No seven-page Telea rows for {side}')
    for r in rows:
        if not r['mask_hash']:
            a=np.load(r['mask_path'],mmap_mode='r',allow_pickle=False); r['mask_hash']=bh(a); r['mask_pixels']=int(np.count_nonzero(a)); del a
        r['sig']=(r['page'],r['mask_hash'])
    return rows

C51=resolve_capture('51R2',CAPTURE_51_OVERRIDE); C53=resolve_capture('53R2',CAPTURE_53_OVERRIDE)
req(C51.resolve()!=C53.resolve(),'Capture roots are identical')
L=index(C51,'51R2'); R=index(C53,'53R2')
common=Counter(x['sig'] for x in L)&Counter(x['sig'] for x in R)
def exceptional(rows):
    rem=Counter(common); out=[]
    for r in sorted(rows,key=lambda x:(x['page'],x['call_number'],str(x['call_dir']))):
        if rem[r['sig']]>0: rem[r['sig']]-=1
        else: out.append(r)
    req(sum(rem.values())==0,'Common multiset removal failed'); return out
EL,ER=exceptional(L),exceptional(R)
print('shared',sum(common.values()),'exceptional',len(EL),len(ER))
req(len(EL)==EXPECTED and len(ER)==EXPECTED,f'Expected 9+9 exceptional masks; got {len(EL)}+{len(ER)}')
req(Counter(x['page'] for x in EL)==Counter(x['page'] for x in ER),'Exceptional page populations differ')

def loadmask(r): return np.asarray(np.load(r['mask_path'],mmap_mode='r',allow_pickle=False)>0)
pairs=[]; costs=[]
for page in sorted(PAGES):
    a=[x for x in EL if x['page']==page]; b=[x for x in ER if x['page']==page]
    if not a: continue
    A=[loadmask(x) for x in a]; B=[loadmask(x) for x in b]; req(len({x.shape for x in A+B})==1,f'{page} shape mismatch')
    C=np.array([[np.count_nonzero(x!=y) for y in B] for x in A],dtype=np.int64)
    for i,x in enumerate(a):
        for j,y in enumerate(b): costs.append({'page':page,'call_51':x['call_number'],'call_53':y['call_number'],'pixels_51':int(A[i].sum()),'pixels_53':int(B[j].sum()),'xor':int(C[i,j])})
    ii,jj=linear_sum_assignment(C)
    for i,j in zip(ii,jj): pairs.append({'page':page,'l':a[int(i)],'r':b[int(j)],'m51':A[int(i)],'m53':B[int(j)],'xor':int(C[int(i),int(j)])})
req(len(pairs)==EXPECTED,f'Expected 9 pairs; got {len(pairs)}')
pd.DataFrame(costs).to_csv(TABLES/'all_page_local_pairing_costs.csv',index=False)

# Discover persisted Notebook45 regional M3 root.
def n45_candidates():
    roots=defaultdict(list)
    for p in ROOT.rglob('M3.png'):
        pg=pageof(p)
        if pg not in PAGES: continue
        parts=list(p.parts)
        for i,v in enumerate(parts):
            if v.lower()=='masks' and i>0: roots[Path(*parts[:i])].append(p)
    out=[]
    for r,paths in roots.items():
        if PAGES.issubset({pageof(p) for p in paths}): out.append((len(paths),r.stat().st_mtime,r))
    return sorted(out)
if NOTEBOOK45_ROOT_OVERRIDE:
    N45=Path(NOTEBOOK45_ROOT_OVERRIDE); req(N45.exists(),f'Bad Notebook45 override: {N45}')
else:
    cand=n45_candidates(); [print('N45 candidate',x[2],'files',x[0]) for x in cand]
    req(cand,'No Notebook45 root with M3.png for all seven pages. Set NOTEBOOK45_ROOT_OVERRIDE.')
    N45=cand[-1][2]
print('N45:',N45)

def rid(path,page):
    parts=list(path.parts); pos=[i for i,v in enumerate(parts) if pageof(v)==page]
    if pos and pos[-1]+1<len(parts)-1:return parts[pos[-1]+1]
    return path.parent.name
regions=defaultdict(list); inv=[]
for p in sorted(N45.rglob('M3.png')):
    pg=pageof(p)
    if pg not in PAGES: continue
    a=cv2.imread(str(p),0)
    if a is None: continue
    m=a>0; region=str(rid(p,pg)); regions[pg].append({'id':region,'path':p,'mask':m}); inv.append({'page':pg,'region_id':region,'path':str(p),'pixels':int(m.sum()),'hash':ah(m.astype(np.uint8))})
for pg in PAGES:req(regions[pg],f'{pg}: no regional M3 masks')
pd.DataFrame(inv).to_csv(AUDIT/'notebook45_regional_m3_inventory.csv',index=False)

pair_rows=[]; coords=[]; ownersummary=[]
for pid,p in enumerate(pairs,1):
    pg=p['page']; a=p['m51']; b=p['m53']
    for z in regions[pg]: req(z['mask'].shape==a.shape,f'{pg}/{z["id"]} shape mismatch')
    shared=a&b; o51=a&~b; o53=b&~a; xor=a^b
    label=f'{pg}_pair{pid:02d}_c51_{p["l"]["call_number"]}_c53_{p["r"]["call_number"]}'
    d=IMAGES/label; d.mkdir()
    for name,m in [('mask_51',a),('mask_53',b),('shared',shared),('only_51',o51),('only_53',o53),('xor',xor)]: cv2.imwrite(str(d/f'{name}.png'),m.astype(np.uint8)*255)
    pair_rows.append({'pair_id':pid,'pair_label':label,'page':pg,'call_51':p['l']['call_number'],'call_53':p['r']['call_number'],
        'context_region_51':p['l']['region_id'],'context_region_53':p['r']['region_id'],'mask_path_51':str(p['l']['mask_path']),'mask_path_53':str(p['r']['mask_path']),
        'pixels_51':int(a.sum()),'pixels_53':int(b.sum()),'shared':int(shared.sum()),'only_51':int(o51.sum()),'only_53':int(o53.sum()),'xor':int(xor.sum())})
    for direction,diff in [('ONLY_51',o51),('ONLY_53',o53)]:
        oc=Counter()
        for y,x in zip(*np.where(diff)):
            own=sorted(z['id'] for z in regions[pg] if z['mask'][y,x]); text='|'.join(own); oc[text]+=1
            coords.append({'pair_id':pid,'pair_label':label,'page':pg,'call_51':p['l']['call_number'],'call_53':p['r']['call_number'],'direction':direction,'y':int(y),'x':int(x),'owner_count':len(own),'owner_ids':text,'single_owner':len(own)==1})
        for text,n in oc.items(): ownersummary.append({'pair_id':pid,'pair_label':label,'page':pg,'direction':direction,'owner_ids':text,'owner_count':0 if not text else len(text.split('|')),'pixels':n})
P=pd.DataFrame(pair_rows).sort_values(['page','pair_id']); C=pd.DataFrame(coords).sort_values(['page','pair_id','direction','y','x']); O=pd.DataFrame(ownersummary)
P.to_csv(TABLES/'exceptional_mask_pairs.csv',index=False); C.to_csv(TABLES/'exceptional_mask_coordinate_differences.csv',index=False); O.to_csv(TABLES/'exceptional_coordinate_ownership_summary.csv',index=False)
display(P); display(O)
req(len(C)>0,'No exceptional coordinate differences')
single=bool(C.single_owner.all()); zero=int((C.owner_count==0).sum()); multi=int((C.owner_count>1).sum())
s53=bool((P.only_51==0).all() and (P.only_53>0).all()); s51=bool((P.only_53==0).all() and (P.only_51>0).all())
if s53 and single: diag='53R2_EXCEPTIONAL_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_51R2'; fix='Remove the ONLY_53 coordinates from the nine 53R2 regional calls by reproducing 51R2 regional assignment.'
elif s51 and single: diag='51R2_EXCEPTIONAL_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_53R2'; fix='Add the missing ONLY_51 coordinates to the corresponding 53R2 regional calls.'
elif single: diag='BIDIRECTIONAL_SINGLE_OWNER_REGIONAL_ASSIGNMENT_DIFFERENCE'; fix='Reproduce 51R2 regional assignment exactly; the nine masks contain additions and omissions.'
elif zero: diag='DIFFERING_COORDINATES_NOT_COVERED_BY_NOTEBOOK45_M3'; fix='Resolve Notebook45 lineage before editing 53R2.'
else: diag='REGIONAL_OWNERSHIP_AMBIGUITY_REMAINS'; fix='Resolve multi-owner regional assignment before editing 53R2.'
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'reuses_persisted_masks':True,'regenerates_masks':False,'capture_51':str(C51),'capture_53':str(C53),'notebook45_root':str(N45),'shared_calls':sum(common.values()),'exceptional_51':len(EL),'exceptional_53':len(ER),'pairs':len(P),'only_51_coordinates':int((C.direction=='ONLY_51').sum()),'only_53_coordinates':int((C.direction=='ONLY_53').sum()),'all_single_owner':single,'zero_owner':zero,'multi_owner':multi,'diagnosis':diag,'first_correction':fix,'stage_f_pass_condition':'After correcting 53R2, rerun Notebook54 and require zero corrected-page differences.','software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'18_mask_coordinate_comparator_verdict.json').write_text(json.dumps(verdict,indent=2))
(OUT/'README.txt').write_text('Start with 18_mask_coordinate_comparator_verdict.json and the three CSV files in tables/. This run reused saved mask.npy files and did not regenerate masks.\n')
print(json.dumps(verdict,indent=2)); print('COMPLETE:',OUT)
